![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E02 - StrOutputParser: extraer el texto automaticamente (Resolution)

## BLOQUE 1 — ¿Qué es un OutputParser y por qué existe?

### El problema

Cuando llamás a `llm.invoke()`, LangChain no devuelve un string — devuelve un objeto `AIMessage`:

```python
result = llm.invoke("Hola")
print(type(result))  # <class 'langchain_core.messages.ai.AIMessage'>
```

Esto es intencional: `AIMessage` guarda **metadatos** además del texto (tokens usados, ID del mensaje, modelo, etc.).

Para usar el texto necesitás hacer `result.content`. Pero si tenés 20 funciones que llaman al LLM, escribir `.content` en cada una es:
- **Repetitivo**: código duplicado
- **Frágil**: si la estructura interna cambia, hay que modificar 20 lugares
- **Propenso a errores**: es fácil olvidar el `.content` y pasar un `AIMessage` donde se espera un `str`

**`StrOutputParser`** resuelve esto: se pone al final de la chain y extrae el `.content` automáticamente.

### El rol del OutputParser en el pipeline

```text
Input -> PromptTemplate -> LLM -> OutputParser -> Respuesta (string)
                                  ^^^^^^^^^^^^
                          Normaliza la salida del modelo
```

El parser es el **último eslabón**: garantiza que el tipo de salida sea siempre el mismo, sin importar el modelo ni el proveedor.

## BLOQUE 2 — Tipos de OutputParsers en LangChain

LangChain tiene varios parsers para distintos formatos de salida:

| Parser | Entrada | Salida | ¿Cuándo usarlo? |
|---|---|---|---|
| `StrOutputParser` | `AIMessage` | `str` | Respuestas de texto libre (chat, preguntas, resúmenes) |
| `JsonOutputParser` | `AIMessage` | `dict` / `list` | Cuando el modelo devuelve JSON estructurado |
| `PydanticOutputParser` | `AIMessage` | Objeto Pydantic | Cuando necesitás validar contra un schema estricto |
| `CommaSeparatedListOutputParser` | `AIMessage` | `List[str]` | Para listas simples separadas por comas |
| `DatetimeOutputParser` | `AIMessage` | `datetime` | Cuando el modelo devuelve fechas |
| `BooleanOutputParser` | `AIMessage` | `bool` | Para clasificación sí/no |
| `EnumOutputParser` | `AIMessage` | `Enum` | Cuando la respuesta debe ser un valor de un conjunto fijo |

**En este notebook** nos enfocamos en `StrOutputParser`, el más común y simple.

## BLOQUE 3 — Setup inicial

In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    CommaSeparatedListOutputParser,
)
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

## BLOQUE 4 — Sin parser: el problema del AIMessage

`llm.invoke()` devuelve un `AIMessage`. No es un string. Hay que acceder a `.content`.

In [ ]:
result_raw = llm.invoke("Di solo: hola")

print(f"Tipo: {type(result_raw).__name__}")
print(f"result_raw: {result_raw}")
print()
print(f"Para obtener el texto: result_raw.content = '{result_raw.content}'")
print()
print("--- Metadatos dentro del AIMessage ---")
print(f"response_metadata: {result_raw.response_metadata}")
print(f"id: {result_raw.id}")
if hasattr(result_raw, 'usage_metadata') and result_raw.usage_metadata:
    print(f"usage_metadata: {result_raw.usage_metadata}")

### ¿Por qué AIMessage existe y no es solo un string?

LangChain envuelve la respuesta en `AIMessage` por 3 razones:

1. **Metadatos de tracing**: cuántos tokens consumiste (`usage_metadata`), qué modelo respondió, latencia
2. **Herramientas (tool_calls)**: cuando el modelo quiere llamar una tool, la información va en `AIMessage.tool_calls`
3. **Interfaz unificada**: todos los proveedores (OpenAI, Anthropic, Google) devuelven `AIMessage` con la misma estructura

```text
AIMessage
  |-- content: str               <- el texto de la respuesta
  |-- tool_calls: List[dict]     <- si el modelo quiere invocar una herramienta
  |-- response_metadata: dict    <- modelo, tokens, etc.
  |-- id: str                    <- identificador único del mensaje
  |-- usage_metadata: dict       <- input_tokens, output_tokens
```

**Problema**: si tu código solo necesita el texto, tener que navegar `.content` en cada lugar es tedioso. Ahí entra `StrOutputParser`.

## BLOQUE 5 — Con StrOutputParser: uso directo

`StrOutputParser` toma un `AIMessage` y devuelve el `content` como string. Se puede usar solo o componerlo.

In [ ]:
# El parser recibe el AIMessage y devuelve el string
result_parsed = parser.invoke(result_raw)
print(f"Tipo con parser directo: {type(result_parsed).__name__}")
print(f"Valor: '{result_parsed}'")
print()
print("StrOutputParser.invoke(aimessage) -> str")

## BLOQUE 6 — TODO 1: Composición con LCEL (`llm | parser`)

Lo más común es poner el parser al final de la chain con el operador `|`. Así la chain completa devuelve un string directamente.

In [ ]:
# TODO 1: componer llm | parser
chain_con_parser = llm | parser
print(f"Tipo de la chain: {type(chain_con_parser).__name__}")
print()
print("Ahora .invoke() devuelve str directamente")

## BLOQUE 7 — TODO 2: Invocar y comparar tipos

Compará el resultado de `llm.invoke()` vs `chain_con_parser.invoke()`.

In [ ]:
# TODO 2: invocar y comparar
result_chain = chain_con_parser.invoke("Di solo: hola")
print(f"Con llm solo:         tipo={type(result_raw).__name__}")
print(f"Con llm | parser:     tipo={type(result_chain).__name__}")
print(f"Valor: '{result_chain}'")
print()
print("Con el parser en la chain, el resultado final SIEMPRE es un string")

## BLOQUE 8 — Ejemplos avanzados con otros parsers

### 8.1 StrOutputParser en una chain completa con PromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template("Resume en 1 oracion: {texto}")
chain_completa = prompt | llm | parser

resultado = chain_completa.invoke({"texto": "LangChain es un framework para orquestar LLMs."})
print(f"Chain completa -> tipo: {type(resultado).__name__}")
print(f"Resultado: {resultado}")

### 8.2 JsonOutputParser — cuando el modelo devuelve JSON

Si le pedís al modelo que devuelva JSON, `JsonOutputParser` parsea automáticamente.

In [ ]:
prompt_json = ChatPromptTemplate.from_template(
    "Devuelve un JSON con 'nombre' y 'edad' para: {persona}"
)
json_parser = JsonOutputParser()
chain_json = prompt_json | llm | json_parser

resultado_json = chain_json.invoke({"persona": "Juan Perez, 30 anos"})
print(f"Tipo: {type(resultado_json).__name__}")
print(f"Valor: {resultado_json}")
print(f"Acceso por clave: nombre={resultado_json['nombre']}, edad={resultado_json['edad']}")

### 8.3 CommaSeparatedListOutputParser — listas simples

Cuando necesitás que el modelo devuelva una lista de items.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

list_parser = CommaSeparatedListOutputParser()

prompt_lista = ChatPromptTemplate.from_template(
    "Lista 3 {categoria} separadas por coma.\n{formato_instrucciones}"
)

chain_lista = prompt_lista | llm | list_parser
resultado_lista = chain_lista.invoke({
    "categoria": "frutas tropicales",
    "formato_instrucciones": list_parser.get_format_instructions()
})
print(f"Tipo: {type(resultado_lista).__name__}")
print(f"Valor: {resultado_lista}")
print(f"Cantidad de items: {len(resultado_lista)}")

## BLOQUE 9 — Debugging del parser

Podés inspeccionar el parser y ver qué hace con distintos inputs.

In [ ]:
print("========== DEBUG StrOutputParser ==========")
print(f"Tipo: {type(parser).__name__}")
print(f"Input esperado: AIMessage o string")

# Probar con AIMessage
msg = AIMessage(content="texto de prueba")
print(f"\nInput: AIMessage(content='{msg.content}')")
print(f"Output: '{parser.invoke(msg)}'")
print(f"Tipo output: {type(parser.invoke(msg)).__name__}")

# Probar con string directo (también funciona)
print(f"\nInput: str('texto directo')")
print(f"Output: '{parser.invoke('texto directo')}'")
print("==========================================")

## BLOQUE 10 — Comparación final

| Situación | Sin parser | Con `StrOutputParser` |
|---|---|---|
| **Llamada** | `llm.invoke(q)` | `(llm \| parser).invoke(q)` |
| **Resultado** | `AIMessage` | `str` |
| **Para obtener texto** | `resultado.content` | Directo |
| **Tool calls** | `resultado.tool_calls` | No aplica (el parser solo extrae texto) |
| **Riesgo** | Olvidar `.content` y pasar AIMessage donde va str | Siempre str |
| **Flexibilidad** | Acceso a metadatos | Solo texto (lo que necesitás el 90% del tiempo) |

### ¿Cuándo NO usar StrOutputParser?

- Cuando necesitás los **metadatos** (tokens, modelo, ID) — ahí usá `llm` solo
- Cuando el modelo debe devolver **JSON estructurado** — usá `JsonOutputParser` o `PydanticOutputParser`
- Cuando el modelo invoca **tools** — necesitás `AIMessage.tool_calls`

### Regla práctica

```text
Si el 90% de tu código solo necesita el texto:
  usa llm | parser en todas tus chains

Si alguna vez necesitás los metadatos:
  usá llm.invoke() directamente solo en ese caso
```

## BLOQUE 11 — Checks automáticos

In [ ]:
def run_checks():
    assert isinstance(llm.invoke("test"), AIMessage), "llm solo debe devolver AIMessage"
    assert chain_con_parser is not None, "TODO 1: chain_con_parser es None"
    result = chain_con_parser.invoke("test")
    assert isinstance(result, str), f"Debe ser str, es {type(result).__name__}"
    assert len(result) > 0, "Resultado vacio"
    print("M3L2 E02 Resolution checks passed")

run_checks()

## Cierre

Hoy aprendiste:

1. **`llm.invoke()` devuelve `AIMessage`**, no un string — por los metadatos que contiene
2. **`StrOutputParser`** extrae `.content` automáticamente, como paso final de la chain
3. **Composición con LCEL**: `llm | parser` hace que `.invoke()` devuelva `str` directamente
4. **Otros parsers**: `JsonOutputParser`, `CommaSeparatedListOutputParser` para formatos estructurados
5. **Debugging**: podés probar el parser con distintos inputs para verificar su comportamiento

### Próximos pasos

- **E03**: chain completa con `prompt | llm | parser`
- **E04**: agregar memoria a la chain con `RunnableWithMessageHistory`
- **E10**: RAG completo donde el parser es el último eslabón del pipeline